# Two-year lung adenocarcinoma prognosis

An executed feasibility study on the patients in Jaksik et al. (2024), Figure 5. The primary question is whether a model trained in TCGA retains its ability to rank mortality risk in CPTAC-3. The second question is whether the reported within-CPTAC-3 score can be exceeded when a random forest is refitted there. All AUCs use death as the positive outcome.

The notebook includes the measurement matrix and the predictions from the completed runs. In Google Colab, use **Runtime → Run all**. A CPU is sufficient. The settings below let you repeat the more expensive fits. The accompanying audit archive includes the data, public-source extraction code, fitted parameter choices, and predictions as ordinary files.

## Models and evaluation

The 32 shared features are 20 methylation probes and 12 RNA measurements. Clinical stage is the ordinal pathological stage I–IV available after staging. The candidate is ridge logistic regression using the shared markers plus stage. The comparisons use stage alone, the shared markers alone with ridge regression, and a molecular random forest.

For external evaluation, all imputation, scaling, and five-fold parameter tuning used TCGA only. We evaluated the fitted models on the published CPTAC-3 set and repeated scoring on all 96 initially eligible patients. We kept the patient as the unit of analysis. For within-cohort evaluation, each patient received a prediction from a model trained on the other nine outer folds. The marker panel had already been selected on TCGA in the paper, which may inflate TCGA estimates.

The paper used DESeq2, R random forests, majority-class downsampling, 100 outer cross-validation repetitions, and inner leave-one-out tuning. Here the expression measure is log₂(TPM + 1), the inner loop is five-fold, and the main forest retains every training patient with class weighting. The 20-repeat comparison includes a downsampled forest on identical outer folds.

In [1]:
# Full reruns use the same functions as the original experiment.
REFIT_EXTERNAL = True
REFIT_INTERNAL = False  # Change to True to repeat the one-split within-cohort fits.
REFIT_STABILITY = False  # Change to True to retrain both forests in all 20 repeats.
WORKERS = 2

In [2]:
#@title Load the bundled observations and reference predictions
import base64, hashlib, io, json, zipfile
import numpy as np
import pandas as pd
encoded = """
...
SwUGAAAAAAYABgB+AQAAQQoBAAAA
"""
archive = base64.b64decode(encoded)
assert hashlib.sha256(archive).hexdigest() == "afa75ef0e1a8b8ba4e67fba3ab894d76014c2a7c090b88a90b26eb2c79cc1d4e"
bundle = zipfile.ZipFile(io.BytesIO(archive))
def bundled_csv(name):
    with bundle.open(name) as source:
        precision = None if name == "patient_matrix.csv" else "round_trip"
        return pd.read_csv(source, index_col="patient_id", float_precision=precision)
data = bundled_csv("patient_matrix.csv")
panels = json.loads(bundle.read("panels.json"))
assert data.index.is_unique
assert len(panels['shared']) == 32 and len(panels['discovery']) == 40
print(data.loc[data.in_paper_analysis].groupby('cohort').death_2y.agg(['count','sum']).rename(columns={'count':'patients','sum':'deaths'}))

         patients  deaths
cohort                   
CPTAC-3        85      10
TCGA          267      89


## Training and scoring functions

Each score is based on out-of-sample predictions. ROC AUC measures ranking, average precision summarizes precision across recall thresholds, and the Brier score measures squared probability error. Higher values are preferable for the first two measures and lower values for Brier score. Bootstrap intervals resample deaths and survivors separately and preserve each patient's predictions across methods.

In [3]:
"""Two-year LUAD model comparison on the paper-listed patients."""
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20261125
PRIMARY = 'Stage + molecular ridge'
FOREST = 'Molecular RF'
MOLECULAR = 'Molecular ridge'
STAGE = 'Stage alone'
DISCOVERY = 'Discovery-panel RF'
DOWNSAMPLED = 'Downsampled molecular RF'

def columns_for(name, panels):
    if name == STAGE:
        return ['stage']
    if name == DISCOVERY:
        return list(panels['discovery'])
    return list(panels['shared']) + (['stage'] if name == PRIMARY else [])

def fit_model(name, train, panels):
    columns = columns_for(name, panels)
    # A feature with no measured value anywhere in this training set cannot enter the model.
    columns = [c for c in columns if train[c].notna().any()]
    steps = [('impute', SimpleImputer(strategy='median'))]
    if name in [FOREST, DISCOVERY, DOWNSAMPLED]:
        weight = None if name == DOWNSAMPLED else 'balanced_subsample'
        model = RandomForestClassifier(n_estimators=300, class_weight=weight,
                                       n_jobs=1, random_state=SEED)
        grid = {'model__max_features': ['sqrt', 0.5, 1.0]}
    else:
        steps.append(('standardize', StandardScaler()))
        model = LogisticRegression(solver='lbfgs', max_iter=2000)
        grid = {'model__C': [0.01, 0.1, 1.0, 10.0]}
    steps.append(('model', model))
    inner = StratifiedKFold(5, shuffle=True, random_state=SEED)
    search = GridSearchCV(Pipeline(steps), grid, scoring='roc_auc', cv=inner, n_jobs=1,
                          error_score='raise')
    search.fit(train[columns], train.death_2y)
    return search, columns

def predictions(name, train, test, panels):
    model, columns = fit_model(name, train, panels)
    return model.predict_proba(test[columns])[:, 1], model.best_params_

def get_scores(y, risk):
    return {'ROC AUC':roc_auc_score(y, risk), 'Average precision':average_precision_score(y, risk),
            'Brier score':brier_score_loss(y, risk)}

def external(data, panels):
    train = data.loc[data.cohort.eq('TCGA')]
    test = data.loc[data.cohort.eq('CPTAC-3')]
    scores = test[['death_2y', 'in_paper_analysis']].copy()
    settings = {}
    for name in [FOREST, STAGE, MOLECULAR, PRIMARY]:
        scores[name], settings[name] = predictions(name, train, test, panels)
    return scores, settings

def performance(scores):
    rows = {name:get_scores(scores.death_2y, scores[name]) for name in scores
            if name not in ['death_2y', 'in_paper_analysis', 'fold']}
    return pd.DataFrame.from_dict(rows, orient='index')

def paired_uncertainty(scores, repeats=10000):
    # For fixed predictions, ROC AUC is the average of all event/non-event pair comparisons.
    # Pair matrices preserve ties. Patient resamples and their multiplicities are shared by models.
    models = [c for c in scores if c not in ['death_2y', 'in_paper_analysis', 'fold']]
    values = scores[models].to_numpy()
    events, survivors = values[scores.death_2y.eq(1)], values[scores.death_2y.eq(0)]
    gap = events[:,None,:] - survivors[None,:,:]
    wins = (gap > 0).astype(float) + 0.5 * (gap == 0)
    rng = np.random.default_rng(SEED)
    w_event = rng.multinomial(len(events), np.full(len(events), 1/len(events)), size=repeats)
    w_surv = rng.multinomial(len(survivors), np.full(len(survivors), 1/len(survivors)), size=repeats)
    auc_samples = np.einsum('ri,ijm,rj->rm',w_event,wins,w_surv,optimize=True)/(len(events)*len(survivors))
    points = wins.mean(axis=(0,1))
    assert np.allclose(points,[roc_auc_score(scores.death_2y,scores[m]) for m in models])
    intervals = {m: [points[i], *np.quantile(auc_samples[:,i],[.025,.975])]
                 for i,m in enumerate(models)}
    for other in [FOREST, STAGE, MOLECULAR]:
        a, b = models.index(PRIMARY), models.index(other)
        intervals[f'{PRIMARY} minus {other}'] = [points[a]-points[b], *np.quantile(auc_samples[:,a]-auc_samples[:,b],[.025,.975])]
    return pd.DataFrame.from_dict(intervals, orient='index', columns=['estimate','lower_95','upper_95'])

def fold_prediction(name, fold, train_rows, test_rows, cohort, panels):
    pred, params = predictions(name, cohort.iloc[train_rows], cohort.iloc[test_rows], panels)
    return name, fold, cohort.index[test_rows], pred, params

def internal(cohort, panels, names, workers=2):
    folds = list(StratifiedKFold(10, shuffle=True, random_state=SEED).split(cohort,cohort.death_2y))
    jobs = (delayed(fold_prediction)(name,k,tr,te,cohort,panels)
            for name in names for k,(tr,te) in enumerate(folds,1))
    fitted = Parallel(n_jobs=workers)(jobs)
    result = cohort[['death_2y']].copy()
    settings = []
    for name,k,ids,risk,params in fitted:
        result.loc[ids,'fold'] = k
        result.loc[ids,name] = risk
        settings.append({'model':name,'fold':k,**params})
    assert result[names].notna().all().all()
    return result, settings

## External test on CPTAC-3

The primary analysis uses 85 patients, including 10 deaths. The 96-patient sensitivity analysis also includes the 11 cases the original paper dropped for missing methylation. Their published outcomes are available in Table S1. These two results have overlapping patients.

In [4]:
if REFIT_EXTERNAL:
    ext, settings = external(data, panels)
    print('Parameters chosen on TCGA:', settings)
else:
    ext = bundled_csv('external_predictions.csv')
main = ext.loc[ext.in_paper_analysis]
print('Published 85 patients')
print(performance(main).round(4).to_string())
print('Paired, stratified patient-bootstrap intervals')
print(paired_uncertainty(main).round(4).to_string())
print('All 96 eligible patients')
print(performance(ext).round(4).to_string())

Parameters chosen on TCGA: {'Molecular RF': {'model__max_features': 'sqrt'}, 'Stage alone': {'model__C': 0.01}, 'Molecular ridge': {'model__C': 0.01}, 'Stage + molecular ridge': {'model__C': 0.1}}
Published 85 patients
                         ROC AUC  Average precision  Brier score
Molecular RF              0.4613             0.2589       0.1808
Stage alone               0.7880             0.3268       0.1382
Molecular ridge           0.4507             0.1275       0.1845
Stage + molecular ridge   0.4653             0.1221       0.2625
Paired, stratified patient-bootstrap intervals
                                               estimate  lower_95  upper_95
Molecular RF                                     0.4613    0.2413    0.6900
Stage alone                                      0.7880    0.6493    0.9060
Molecular ridge                                  0.4507    0.2653    0.6453
Stage + molecular ridge                          0.4653    0.2733    0.6587
Stage + molecular ridge minus

## Within-cohort comparison with Figure 5

The detailed figure reports TCGA AUC 0.839 for all 40 features and 0.830 for 32 shared features. CPTAC-3 AUC is 0.815 for a classifier fitted within that cohort. One discovery-only probe, cg20673929, was entirely missing in the reconstructed TCGA matrix. The discovery-panel forest therefore used 39 measurable features. Three TCGA patients lacked matched 450K methylation and two lacked a recorded primary pathological stage. The fitted training medians supplied missing values. The primary shared-panel analysis has observations for all 32 marker columns.

In [5]:
for cohort_name, frame in data.loc[data.in_paper_analysis].groupby('cohort',sort=False):
    names = [FOREST,PRIMARY,STAGE,MOLECULAR] + ([DISCOVERY] if cohort_name=='TCGA' else [])
    if REFIT_INTERNAL:
        folds, fold_settings = internal(frame,panels,names,workers=WORKERS)
    else:
        folds = bundled_csv(f'{cohort_name}_oof.csv')
    print(cohort_name)
    print(performance(folds).round(4).to_string())

TCGA
                         ROC AUC  Average precision  Brier score
Molecular RF              0.8072             0.6696       0.1647
Stage + molecular ridge   0.7901             0.6963       0.1644
Stage alone               0.6636             0.4619       0.2112
Molecular ridge           0.7847             0.6989       0.1648
Discovery-panel RF        0.8250             0.6899       0.1589
CPTAC-3
                         ROC AUC  Average precision  Brier score
Molecular RF              0.8320             0.3291       0.0928
Stage + molecular ridge   0.7947             0.4000       0.0925
Stage alone               0.7580             0.3423       0.1018
Molecular ridge           0.7427             0.2727       0.1074


## Does the first CPTAC-3 split generalize to other splits?

This follow-up was specified after observing the first within-cohort AUC. It uses 20 consecutive outer split seeds including the first seed. Both forests use the same outer folds and the same molecular measurements. One keeps all patients and weights the classes within each bootstrap sample. The other uses an unweighted forest after keeping all deaths and sampling 30% of survivors in each outer training fold. Both use five-fold tuning, 300 trees, and the same candidate feature-count settings. The paired comparison evaluates these two training strategies together. Variation between repeats describes sensitivity to splits. All 20 repeats reuse the same 85 people.

In [6]:
from sklearn.model_selection import StratifiedKFold
from joblib import Parallel, delayed
import numpy as np
import pandas as pd

def compare_fold(k, seed, tr, te, cohort, panels):
    train, holdout = cohort.iloc[tr], cohort.iloc[te]
    full, p_full = predictions(FOREST, train, holdout, panels)
    # The paper kept all minority cases and a random 30% of majority cases.
    # Our control applies this rule to each outer training fold before inner tuning.
    events = train.loc[train.death_2y.eq(1)]
    survivors = train.loc[train.death_2y.eq(0)]
    n_keep = round(0.3 * len(survivors))
    sampled = pd.concat([events, survivors.sample(n=n_keep,random_state=seed + k)]).sort_index()
    small, p_small = predictions(DOWNSAMPLED, sampled, holdout, panels)
    out = pd.DataFrame({'seed':seed, 'fold':k, 'death_2y':holdout.death_2y,
                        FOREST:full, DOWNSAMPLED:small,
                        'weighted_mtry':str(p_full['model__max_features']),
                        'downsampled_mtry':str(p_small['model__max_features'])}, index=holdout.index)
    return out

def robustness(cohort, panels, workers=2, repetitions=20):
    jobs = []
    for seed in range(SEED,SEED + repetitions):
        folds = StratifiedKFold(10, shuffle=True, random_state=seed).split(cohort,cohort.death_2y)
        for k,(tr,te) in enumerate(folds,1):
            jobs.append(delayed(compare_fold)(k,seed,tr,te,cohort,panels))
    records = pd.concat(Parallel(n_jobs=workers)(jobs))
    score_rows = []
    for seed, frame in records.groupby('seed'):
        assert len(frame)==len(cohort) and frame.index.is_unique
        metrics = {name:get_scores(frame.death_2y,frame[name])['ROC AUC']
                   for name in [FOREST,DOWNSAMPLED]}
        score_rows.append({'seed':seed, **metrics, 'paired_difference':metrics[FOREST]-metrics[DOWNSAMPLED]})
    return records,pd.DataFrame(score_rows).set_index('seed')

In [7]:
cptac = data.loc[data.cohort.eq('CPTAC-3') & data.in_paper_analysis]
if REFIT_STABILITY:
    predictions_by_repeat, repeat_scores = robustness(cptac, panels, workers=WORKERS, repetitions=20)
else:
    predictions_by_repeat = bundled_csv('repeat_oof_predictions.csv')
    rows = []
    for seed, frame in predictions_by_repeat.groupby('seed'):
        assert frame.index.is_unique and len(frame)==85
        scores = {m:roc_auc_score(frame.death_2y,frame[m]) for m in [FOREST,DOWNSAMPLED]}
        rows.append({'seed':seed, **scores, 'paired_difference':scores[FOREST]-scores[DOWNSAMPLED]})
    repeat_scores = pd.DataFrame(rows).set_index('seed')
print(repeat_scores.round(4).to_string())
print('Summary over all 20 splits')
print(repeat_scores.agg(['mean','median','min','max']).round(4).to_string())
print('Weighted forest repeats reaching 0.816:', int((repeat_scores[FOREST] >= .816).sum()), 'of', len(repeat_scores))
print('AUC pooling all held-out predictions across repeats:')
for m in [FOREST,DOWNSAMPLED]:
    print(m, round(roc_auc_score(predictions_by_repeat.death_2y,predictions_by_repeat[m]),4))

          Molecular RF  Downsampled molecular RF  paired_difference
seed                                                               
20261125        0.8320                    0.8220             0.0100
20261126        0.8527                    0.8413             0.0113
20261127        0.8200                    0.7607             0.0593
20261128        0.8040                    0.8013             0.0027
20261129        0.8413                    0.7733             0.0680
20261130        0.8180                    0.8080             0.0100
20261131        0.8640                    0.7993             0.0647
20261132        0.8447                    0.8107             0.0340
20261133        0.8260                    0.8853            -0.0593
20261134        0.8487                    0.8220             0.0267
20261135        0.8400                    0.7680             0.0720
20261136        0.8360                    0.7793             0.0567
20261137        0.8073                    0.8027

## What the experiment can establish

Each CPTAC-3 split has ten deaths and 75 survivors. With fixed scores, moving one death past one survivor changes the untied empirical AUC by 1/750, approximately 0.00133. The paper's original per-fold predictions and complete final classifier driver were unavailable from the checked public repository. A paired superiority estimate against the published 0.815 therefore cannot be recovered from the published AUC alone. The bootstrap intervals above apply to the locked external predictions, conditional on the fitted models.

One follow-up hypothesis is that a model anchored to pathological stage can accept separate methylation and RNA scores selectively when each assay adds information across training centers. The decisive experiment would lock the panel, coefficients, and preprocessing in development data, then measure incremental AUC and calibration against stage alone in a new cohort with both assays and a prespecified two-year endpoint. The original TCGA and CPTAC outcome rules differ for non-cancer deaths. Harmonize the event and censoring definitions for that study.

## Data provenance

Patient IDs and labels come from supplementary Table S1, candidate discovery markers from Table S2, and the analyzed IDs and shared marker names from Table S3. Only Table S3 identifiers and column names were used. Patient-level SHAP values are derived from a fitted model and were excluded from every predictor matrix. Tumor RNA and methylation were queried independently from the UCSC Xena GDC hub. GDC biospecimen metadata identified primary tumor specimens and primary-diagnosis pathological stage. When multiple primary tumor samples matched, their measurements were combined using the per-patient median.

- Jaksik et al., *Multiomics-Based Feature Extraction and Selection for the Prediction of Lung Cancer Survival* (2024). [Article and supplementary data](https://doi.org/10.3390/ijms25073661)
- Author scripts inspected at commit 08c7ead9765309c85b0ff1d660f1ff5b41507af3. [Repository](https://github.com/rjaksik/LUAD_multiomic/tree/08c7ead9765309c85b0ff1d660f1ff5b41507af3)
- Public data and processing. [NCI GDC](https://portal.gdc.cancer.gov/) · [UCSC Xena GDC ETL](https://github.com/ucscXena/xena-GDC-ETL)
- Related stage-benchmarked work on a different RNA panel. [Cross-platform validation of a locked 12-gene panel](https://pmc.ncbi.nlm.nih.gov/articles/PMC13704482/)

Executed with Python 3.12.14, scikit-learn 1.8.0, pandas 2.2.3, NumPy 2.3.5 and joblib 1.5.3. Other versions may yield slightly different fits.